# 3regions study ASR workflow
 
 This notebook follows the same deterministic and uncertainty ASR workflow used in the UNCASExt examples, with a custom 3 region aggregation setup.
 
 ## Required aggregation CSVs for this case
 Copy the five files from `aggregation_csvs/` into the corresponding workspace folders before running:
 
 - exiobase_agg/agg_reg_elec_3regions_study.csv -> data_raw/mrio/exiobase_3/aggregation/agg_reg_elec_3regions_study.csv
 - exiobase_agg/agg_reg_oecd_d_3regions_study.csv -> data_raw/mrio/exiobase_3/aggregation/agg_reg_oecd_d_3regions_study.csv
 - exiobase_agg/agg_sec_oecd_d_3regions_study.csv -> data_raw/mrio/exiobase_3/aggregation/ixi/agg_sec_oecd_d_3regions_study.csv
 - exiobase_agg/agg_sec_elec_3regions_study.csv -> data_raw/mrio/exiobase_3/aggregation/ixi/agg_sec_elec_3regions_study.csv
 - oecd_v2025_agg/agg_reg_oecd_d_3regions_study.csv -> data_raw/mrio/oecd_v2025/aggregation/agg_reg_oecd_d_3regions_study.csv
 
 For full context and output notes, read README.md in this folder.
 
 All five aggregation files are required before processing and ASR uncertainty execution.


In [ ]:
from pathlib import Path

from pyaesa import (
    set_workspace,
    download_pop_gdp,
    download_mrio,
    download_ar6,
    process_pop_gdp,
    process_mrio,
    deterministic_asocc,
    disaggregate_asocc,
    uncertainty_asr,
)

# Replace this placeholder with the pyaesa workspace to use for the study.
WORKSPACE_ROOT = Path("/path/to/pyaesa_workspace")
set_workspace(WORKSPACE_ROOT)

YEARS_ASR = list(range(1995, 2023))
PROJECT_NAME = "3regions_study"
FU_CODE = "L2.c.b"
R_C = ["EU27", "BRICS5", "USMCA"]
S_D = ["D"]
S_ELEC = ["Electricity"]

## Data download

In [ ]:
download_pop_gdp()
download_ar6()
download_mrio("exiobase_3102_ixi")
download_mrio("oecd_v2025")


## Data processing

Create all required processed MRIO and population/GDP outputs for the study.


In [ ]:
process_pop_gdp()


In [ ]:
process_mrio(
    source="exiobase_3102_ixi",
    years=YEARS_ASR,
    agg_sec=True,
    agg_reg=True,
    agg_version="elec_3regions_study",
    lcia_method=["pb_lcia", "gwp100_lcia"],
)


In [ ]:
process_mrio(
    source="exiobase_3102_ixi",
    years=YEARS_ASR,
    agg_sec=True,
    agg_reg=True,
    agg_version="oecd_d_3regions_study",
)


In [ ]:
process_mrio(
    source="exiobase_3102_ixi",
    years=YEARS_ASR,
    lcia_method=["pb_lcia", "gwp100_lcia"],
)

In [ ]:
process_mrio(
    source="oecd_v2025",
    years=YEARS_ASR,
    agg_reg=True,
    agg_version="oecd_d_3regions_study",
)


## Deterministic aSoCC

Run OECD D, EXIOBASE D, and Electricity runs with LCIA where requested.


In [ ]:
deterministic_asocc(
    project_name=PROJECT_NAME,
    source="oecd_v2025",
    agg_reg=True,
    agg_version="oecd_d_3regions_study",
    years=YEARS_ASR,
    fu_code=FU_CODE,
    s_p=S_D,
    r_c=R_C,
    figures=False,
)


In [ ]:
deterministic_asocc(
    project_name=PROJECT_NAME,
    source="exiobase_3102_ixi",
    agg_reg=True,
    agg_sec=True,
    agg_version="oecd_d_3regions_study",
    years=YEARS_ASR,
    fu_code=FU_CODE,
    s_p=S_D,
    r_c=R_C,
    figures=False,
)


In [ ]:
deterministic_asocc(
    project_name=PROJECT_NAME,
    source="exiobase_3102_ixi",
    agg_reg=True,
    agg_sec=True,
    agg_version="elec_3regions_study",
    years=YEARS_ASR,
    fu_code=FU_CODE,
    s_p=S_ELEC,
    r_c=R_C,
    lcia_method=["pb_lcia", "gwp100_lcia"],
    figures=False,
)


## Disaggregation

Build inter-MRIO disaggregated source `oecd_electricity` for uncertainty.


In [ ]:
disaggregate_asocc(
    disaggregation_config={
        "target_agg_run": {
            "source": "oecd_v2025",
            "agg_reg": True,
            "agg_version": "oecd_d_3regions_study",
            "s_p": S_D,
        },
        "ref_agg_run": {
            "source": "exiobase_3102_ixi",
            "agg_reg": True,
            "agg_sec": True,
            "agg_version": "oecd_d_3regions_study",
            "s_p": S_D,
        },
        "ref_disagg_run": {
            "source": "exiobase_3102_ixi",
            "agg_reg": True,
            "agg_sec": True,
            "agg_version": "elec_3regions_study",
            "s_p": S_ELEC,
        },
        "disaggregation_specs": [
            {"agg_sector_label": "D", "disagg_sector_label": "Electricity"},
        ],
        "new_disagg_version_name": "oecd_electricity",
    },
    base_asocc_args={
        "project_name": PROJECT_NAME,
        "years": YEARS_ASR,
        "fu_code": FU_CODE,
        "r_c": R_C,
    },
    figures=False,
)


## ASR uncertainty

ASR uncertainty runs using for Phase A IO-LCA.

In [ ]:
uncertainty_config_common = {
    "asocc_uncertainty_sources": {
        "inter_mrio_uncertainty": {
            "active": True,
            "alternate_source": "oecd_electricity",
        },
    },
}
base_asocc_args = {}
uncertainty_asr(
    project_name=PROJECT_NAME,
    source="exiobase_3102_ixi",
    agg_reg=True,
    agg_sec=True,
    agg_version="elec_3regions_study",
    years=YEARS_ASR,
    fu_code=FU_CODE,
    s_p=S_ELEC,
    r_c=R_C,
    lcia_method="pb_lcia",
    lca_args={"io_lca": {"active": True}},
    base_asocc_args={},
    uncertainty_config=uncertainty_config_common,
    subfigures=False,
    sobol_parameters={"active": False},
)


In [ ]:
uncertainty_asr(
    project_name=PROJECT_NAME,
    source="exiobase_3102_ixi",
    agg_reg=True,
    agg_sec=True,
    agg_version="elec_3regions_study",
    years=list(range(2000, 2023)),
    fu_code=FU_CODE,
    s_p=S_ELEC,
    r_c=R_C,
    lcia_method="gwp100_lcia",
    base_cc_args={
        "static": {"active": False},
        "dynamic_ar6": {
            "active": True,
            "ssp_scenario": "SSP2",
        },
    },
    uncertainty_config={
        **uncertainty_config_common,
        "ar6_cc_uncertainty_sources": {
            "dynamic_ar6_cc_uncertainty": {
                "active": True,
                "category_uncertainty": True,
            },
        },
    },
    lca_args={"io_lca": {"active": True}},
    base_asocc_args={},
    subfigures=False,
    sobol_parameters={"active": False},
)
